# LoopedQwen — Experiment 004: normalized projected loop updates

Этот notebook принимает ZIP с кодовой базой, готовит FineWeb, обучает контроль и две абляции, выполняет evaluation и скачивает компактный ZIP для анализа.

Главное отличие от Experiment 003: долгие Python-стадии выполняются **в текущем Jupyter kernel**, не в дочернем процессе. Поэтому `tqdm.auto`, ETA, validation progress и training metrics отображаются непосредственно в активной ячейке.

Перед запуском выберите **Runtime → Change runtime type → T4 GPU** (или более мощную GPU).


In [ ]:
# 1. Загрузите ZIP с кодовой базой Experiment 004
from google.colab import files
from pathlib import Path
from zipfile import ZipFile
import os, shutil

uploaded = files.upload()
zip_names = [name for name in uploaded if name.lower().endswith('.zip')]
if len(zip_names) != 1:
    raise ValueError(f'Ожидался ровно один ZIP, получено: {zip_names}')

extract_dir = Path('/content/loopedqwen_exp004')
if extract_dir.exists():
    shutil.rmtree(extract_dir)
extract_dir.mkdir(parents=True)
with ZipFile(zip_names[0]) as archive:
    archive.extractall(extract_dir)

projects = list(extract_dir.rglob('pyproject.toml'))
if len(projects) != 1:
    raise RuntimeError(f'Не удалось однозначно найти корень проекта: {projects}')
repo_root = projects[0].parent
os.chdir(repo_root)
print('Repository root:', repo_root)
print('Experiment 004 files:')
for path in sorted((repo_root / 'experiments/004_normalized_loop_updates').rglob('*')):
    if path.is_file():
        print(' •', path.relative_to(repo_root))


In [ ]:
# 2. Установка и проверка GPU
%pip install -q -e .

import subprocess, sys, torch
if not torch.cuda.is_available():
    raise RuntimeError('GPU не обнаружена. Включите GPU runtime и перезапустите notebook.')

print('Python:', sys.version.split()[0])
print('Torch:', torch.__version__)
print('GPU:', torch.cuda.get_device_name(0))
runtime = subprocess.run(['nvidia-smi'], text=True, capture_output=True).stdout
Path('runtime_info.txt').write_text(runtime, encoding='utf-8')
print(runtime)


In [ ]:
# 3. Настройки и in-kernel runner
import gc, runpy, time
from IPython.display import HTML, display

VARIANTS = [
    'control_r16',
    'projected_fixed_a025_r16',
    'projected_learned_a025_r16',
]
TRAIN_TOKENS = 10_000_000
VAL_TOKENS = 1_000_000
TOKENIZER_DOCUMENTS = 100_000
EVAL_BATCHES = 50
RESUME_IF_POSSIBLE = True

def stage(title, estimate):
    display(HTML(
        f"<div style='padding:12px 16px;border-left:5px solid #4f46e5;"
        f"background:#eef2ff;margin:8px 0'><b>{title}</b><br>"
        f"<span style='color:#475569'>Грубая оценка для T4: {estimate}. "
        f"После первых итераций tqdm покажет измеренный ETA.</span></div>"
    ))

def run_in_kernel(script, *arguments):
    """Run a repository script in this notebook kernel, without subprocess."""
    script = (repo_root / script).resolve()
    old_argv = sys.argv[:]
    old_cwd = Path.cwd()
    started = time.perf_counter()
    print('\n▶', script.relative_to(repo_root), *map(str, arguments), flush=True)
    try:
        os.chdir(repo_root)
        sys.argv = [str(script), *map(str, arguments)]
        runpy.run_path(str(script), run_name='__main__')
    finally:
        sys.argv = old_argv
        os.chdir(old_cwd)
        gc.collect()
        torch.cuda.empty_cache()
    print(f'✓ Завершено за {(time.perf_counter() - started) / 60:.1f} мин', flush=True)

print('Variants:', VARIANTS)
print('Важно: не закрывайте активную ячейку. Progress и ETA появятся прямо под ней.')


In [ ]:
# 4. Tokenizer и фиксированный FineWeb subset
# Перезапуск безопасен: уже готовые tokenizer/data будут пропущены.
if not Path('tokenizer/tokenizer.json').is_file():
    stage('Tokenizer: 100k документов FineWeb', '10–25 минут, сеть сильно влияет')
    run_in_kernel(
        'scripts/train_tokenizer.py',
        '--output-dir', 'tokenizer',
        '--vocab-size', '16000',
        '--documents', TOKENIZER_DOCUMENTS,
    )
else:
    print('✓ Tokenizer найден — стадия пропущена')

need_data = not Path('data/train.bin').is_file() or not Path('data/val.bin').is_file()
if need_data:
    stage('Подготовка 10M train + 1M validation токенов', '5–15 минут')
    run_in_kernel(
        'scripts/prepare_data.py',
        '--tokenizer', 'tokenizer',
        '--output-dir', 'data',
        '--train-tokens', TRAIN_TOKENS,
        '--val-tokens', VAL_TOKENS,
    )
else:
    print('✓ Token data найдены — стадия пропущена')


In [ ]:
# 5. CPU/GPU sanity checks перед дорогим запуском
stage('Тесты модели и serialization', '1–3 минуты')
import pytest
test_exit = pytest.main(['-q'])
if test_exit != 0:
    raise RuntimeError(f'pytest завершился с кодом {test_exit}')
run_in_kernel('scripts/sanity_check.py')


## План вычислений

Каждый full run делает 305 optimizer steps и не превышает 10M токенов. На T4 один 16-loop run обычно занимает примерно **30–45 минут обучения плюс evaluation**. Три варианта могут занять около **2–3 часов** вместе с подготовкой данных.

Каждый вариант вынесен в отдельную ячейку:

- ячейку можно повторить после разрыва сессии — при `RESUME_IF_POSSIBLE=True` загрузится `last_state.pt`;
- после training автоматически запускается evaluation на 1–32 loops;
- строка training показывает step, ETA, loss, LR, grad norm, tok/s, `α2`, `αR` и последний validation loss.


In [ ]:
# 6A. Контроль: обычный 16-loop residual update
stage('control_r16: training + evaluation', '35–50 минут')
args = [
    '--variant', 'control_r16',
    '--eval-batches', EVAL_BATCHES,
    '--in-process',
]
if RESUME_IF_POSSIBLE:
    args.append('--resume')
run_in_kernel('experiments/004_normalized_loop_updates/run.py', *args)


In [ ]:
# 6B. Fixed gate: normalized update + RMS projection, alpha=0.25
stage('projected_fixed_a025_r16: training + evaluation', '40–55 минут')
args = [
    '--variant', 'projected_fixed_a025_r16',
    '--eval-batches', EVAL_BATCHES,
    '--in-process',
]
if RESUME_IF_POSSIBLE:
    args.append('--resume')
run_in_kernel('experiments/004_normalized_loop_updates/run.py', *args)


In [ ]:
# 6C. Learned schedule: два обучаемых скаляра
stage('projected_learned_a025_r16: training + evaluation', '40–55 минут')
args = [
    '--variant', 'projected_learned_a025_r16',
    '--eval-batches', EVAL_BATCHES,
    '--in-process',
]
if RESUME_IF_POSSIBLE:
    args.append('--resume')
run_in_kernel('experiments/004_normalized_loop_updates/run.py', *args)


In [ ]:
# 7. Итоговая таблица
run_in_kernel('experiments/004_normalized_loop_updates/summarize.py')

import pandas as pd
summary_path = Path('experiments/004_normalized_loop_updates/results/summary.csv')
summary = pd.read_csv(summary_path)
display(summary)

print('\nPPL by loop depth:')
display(summary.pivot(index='eval_loops', columns='variant', values='perplexity').round(2))


In [ ]:
# 8. Создание и скачивание ZIP для анализа
RESULT_ZIP = Path('/content/experiment_004_results.zip')
run_in_kernel(
    'experiments/004_normalized_loop_updates/collect_results.py',
    '--output', RESULT_ZIP,
)
print(f'Готово: {RESULT_ZIP} ({RESULT_ZIP.stat().st_size / 1_000_000:.1f} MB)')
files.download(str(RESULT_ZIP))
